# 01 — Tek bir kaydı okuyup anlamak

Bu derste yalnızca **train dosyasının ilk 5 kaydını** açacağız. Amaç: dosyadaki metin ile Python listesi arasındaki farkı görmek ve kelime–etiket eşleşmesini anlamak.

VS Code kernel: **Python (cmpe586)**. Hücreleri yukarıdan aşağıya çalıştır (`Shift+Enter`). Bu notebook proje kökünden veya `notebooks/` klasöründen çalıştırılabilir. Ham dosyaya hiçbir şey yazılmaz.

Akış: **CSV → pandas tablosu → tek kayıt → Python listeleri → kontroller**.


## 1. CSV dosyasını pandas ile aç

`Path`, dosya yollarını birleştirmemizi sağlar. `pd.read_csv` CSV okuma işini yapar. `nrows=5` yalnızca ilk beş kaydı okur; ilk adımı küçük tutuyoruz.

`DataFrame` satır ve sütunlardan oluşan pandas tablosudur. Buradaki her satır bir **kayıt**; her kayıt birden fazla cümle içerebilir.


In [ ]:
import ast
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data/raw/kurt_cayir_2023/ted_train_tr.csv"
df = pd.read_csv(DATA_PATH, nrows=5, encoding="utf-8")
print("Okunan kayıt sayısı:", len(df))
print("Sütunlar:", df.columns.tolist())
df

## 2. İlk kaydı seç ve hücrenin tipine bak

`iloc[0]`, sıra numarası sıfır olan ilk kaydı seçer. `row["sentence"]` o kaydın kelimeleri içeren hücresidir.

Köşeli parantezlerle görünmesine rağmen hücre henüz bir **string**. CSV bir dosya biçimi; içindeki listelerin Python nesnesine dönüşmesini ayrıca yapacağız.


In [ ]:
row = df.iloc[0]
print("sentence hücresinin Python tipi:", type(row["sentence"]).__name__)
print("tags hücresinin Python tipi:", type(row["tags"]).__name__)
print("sentence hücresi:", row["sentence"])

## 3. Metin olarak saklanan listeleri gerçek listelere çevir

`ast.literal_eval`, bu dosyadaki liste gösterimini Python listesine dönüştürür. Kod çalıştırabilen `eval` kullanmıyoruz. Bozuk bir liste gösterimi varsa dönüşümün hata vermesini isteriz; hatayı gizlemeyiz.

`tokens` şu anda veri setinin kelime konumlarıdır. Transformer tokenizer'ının daha sonra üreteceği **subword** parçaları henüz yok.


In [ ]:
tokens = ast.literal_eval(row["sentence"])
tags = ast.literal_eval(row["tags"])

print("Dönüşüm sonrası tipler:", type(tokens).__name__, type(tags).__name__)
print("Kelime konumu sayısı:", len(tokens))
print("Etiket sayısı:", len(tags))

## 4. Uzunlukları kontrol et ve eşleşmeyi gör

Her kelime konumu için bir noktalama etiketi olmalı. Sayılar farklıysa bir eşleşme hatası vardır ve devam etmemeliyiz. Bu kontrol gerekli ama tek başına yeterli değildir: aynı uzunlukta iki liste yine de yanlış sırada olabilir.

Etiket, **kelimeden sonra** gelecek işarettir:

- `O`: noktalama yok (büyük O harfi).
- `PERIOD`: nokta.
- `COMMA`: virgül.
- `QUESTION_MARK`: soru işareti.

Buradaki `tags`, modelin tahmini değil; veri setindeki öğrenme hedefidir. Kaynak etiketler de hatalı olabilir. Model ileride bu hedeflerle karşılaştırılarak eğitilecek.


In [ ]:
if len(tokens) != len(tags):
    raise ValueError("Kelime ve etiket sayıları eşleşmiyor.")

pairs = pd.DataFrame({"token": tokens, "tag": tags})
pairs

## 5. Küçük görev: iki kontrolü sen ekle

Aşağıdaki hücreye kod yazarak:

1. `tokens` listesindeki boş stringlerin (`""`) **indekslerini** bul ve yazdır.
2. `tags` içindeki dört geçerli sınıfın dışında kalan etiketleri bul ve yazdır.

İpucu: ilk görev için `enumerate(tokens)` hem indeksi hem kelimeyi verir. İkinci görev için `tag not in allowed_tags` kullanabilirsin. `allowed_tags` kümesini aşağıda verdim.

İlk kayıtta ikisinin de boş liste (`[]`) çıkması bekleniyor. Kontrolün boş sonuç vermesi, tüm veri setinin sorunsuz olduğu anlamına gelmez; yalnızca seçtiğimiz kaydı kontrol ediyoruz.

Şimdilik boş kelimeleri **buluyoruz**. Temizleme politikasını belirlemeden silmiyoruz; boş kelimeler noktalama etiketi taşıyabiliyor.


In [ ]:
allowed_tags = {"O", "PERIOD", "COMMA", "QUESTION_MARK"}

# Görev 1: Boş kelimelerin indekslerini bul ve yazdır.

# Görev 2: Geçersiz etiketleri bul ve yazdır.

Bu adım bitince aynı kontrolleri noktalama etiketi taşıyan boş kelimelerin bulunduğu bir kayıtta inceleyeceğiz. Sonra öğrendiğimiz dönüşüm ve kontrolleri tekrar kullanılabilir küçük bir veri okuyucuya taşıyacağız.

Önceden hazırlanmış genel inceleme: `docs/data-audit.md`. Bu notebook, o incelemenin temel adımlarını birlikte öğrenmek içindir.
